# Configuração dos parâmetros

In [0]:
catalog = "olist_medalion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

# Ingestão de Dados

In [0]:
from pyspark.sql.functions import current_timestamp

# Mapeamento dos caminhos no Volume 
path_credits_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"

# Leitura pura
df_credits_tags_raw = spark.read.csv(path_credits_tags, header=True, inferSchema=True)
df_financials_raw = spark.read.csv(path_financials, header=True, inferSchema=True)
df_metrics_raw = spark.read.csv(path_metrics, header=True, inferSchema=True)
df_info_raw = spark.read.csv(path_info, header=True, inferSchema=True)
df_review_raw = spark.read.csv(path_reviews, header=True, inferSchema=True)

## Adicionar coluna ingestion_datetime

In [0]:
# Adição da coluna e criação da tabelas correspondentes
df_credits_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_review_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")

display(spark.table(f"{catalog}.bronze.tb_movies_metrics").limit(5))

## Ingestão da API

In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql.functions import col, to_timestamp

# Cálculo dos últimos 7 dias
hoje = datetime.now()
sete_dias_atras = hoje - timedelta(days=7)

# Formato MM-DD-AAAA
fmt_data_api = "%m-%d-%Y" 
default_inicio = sete_dias_atras.strftime(fmt_data_api)
default_fim = hoje.strftime(fmt_data_api)

# Widgets
dbutils.widgets.text("data_inicio", default_inicio, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", default_fim, "Data Fim (MM-DD-AAAA)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

# Requisição da API
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio_formatada}'&"
    f"@dataFinalCotacao='{data_fim_formatada}'&"
    f"$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url)

In [0]:
# Processamento e verificação dos dados obtidos
if response.status_code == 200: # Fim de semana ou feriado
    dados_json = response.json()
    cotacoes = dados_json.get("value", [])
    
    if cotacoes:
        # Cria o DataFrame Spark
        df_raw = spark.createDataFrame(cotacoes)
        
        # Adiciona timestamp e salva no Delta
        df_bronze = df_raw \
            .withColumn("ingestion_datetime", current_timestamp()) \
            .write.format("Delta").mode("append") \
            .saveAsTable(f"{catalog}.bronze.tb_cotacao_dolar")
        
        print(f"Carga realizada com sucesso! {spark.table(f"{catalog}.bronze.tb_cotacao_dolar").count()} registro(s) inserido(s).")
        display(spark.table(f"{catalog}.bronze.tb_cotacao_dolar").limit(5))
    else:
        print(f"Nenhum dado retornado no período de {data_inicio_formatada} a {data_fim_formatada}.")
else:
    print(f"Erro ao consultar a API. Código de erro HTTP: {response.status_code}")
    print(response.text)